In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("Final Cleaned Data.csv")

In [ ]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 32 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  198 non-null    object 
 1   Course_ID                   198 non-null    object 
 2   Login_Frequency             198 non-null    int64  
 3   Time_Spent_Modules          198 non-null    float64
 4   Participation_Forums        198 non-null    int64  
 5   Quiz_Performance_Average    198 non-null    float64
 6   Assignment_Submissions      198 non-null    int64  
 7   Resource_Access_Frequency   198 non-null    int64  
 8   Session_Duration_Average    198 non-null    float64
 9   Device_Type                 198 non-null    object 
 10  Internet_Bandwidth          198 non-null    float64
 11  Engagement_Level            198 non-null    object 
 12  Hours_Studied               198 non-null    int64  
 13  Attendance                  198 non

In [ ]:
categorical_columns = [
    'Device_Type',
    'Internet_Access',
    'Engagement_Level',
    'Parental_Involvement',
    'Access_to_Resources',
    'Extracurricular_Activities',
    'Motivation_Level',
    'School_Type',
    'Peer_Influence',
    'Learning_Disabilities',
    'Parental_Education_Level',
    'Gender'
]

encoder = LabelEncoder()

for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
def assign_risk(row):
    if row['Exam_Score'] < 60 and row['Attendance'] < 70:
        return 'High'
    elif row['Exam_Score'] < 75 or row['Engagement_Level'] == 0:
        return 'Moderate'
    else:
        return 'Low'

df['Risk_Level'] = df.apply(assign_risk, axis=1)

In [ ]:
X = df[
    [
        'Login_Frequency',
        'Time_Spent_Modules',
        'Participation_Forums',
        'Quiz_Performance_Average',
        'Assignment_Submissions',
        'Resource_Access_Frequency',
        'Session_Duration_Average',
        'Engagement_Level',
        'Hours_Studied',
        'Attendance',
        'Device_Type',
        'Internet_Access',
        'Internet_Bandwidth',
        'Previous_Scores',
        'Sleep_Hours',
        'Tutoring_Sessions',
        'Physical_Activity',
        'Teacher_Quality'
    ]
]

y = df['Risk_Level']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# we look for columns that are still non-numeric
X.dtypes

,0
Login_Frequency,int64
Time_Spent_Modules,float64
Participation_Forums,int64
Quiz_Performance_Average,float64
Assignment_Submissions,int64
Resource_Access_Frequency,int64
Session_Duration_Average,float64
Engagement_Level,int64
Hours_Studied,int64
Attendance,int64


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
X['Teacher_Quality'] = encoder.fit_transform(X['Teacher_Quality'])

/tmp/ipython-input-1655659723.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Teacher_Quality'] = encoder.fit_transform(X['Teacher_Quality'])


In [ ]:
# we recheck which columns are still non-numeric
X.dtypes

,0
Login_Frequency,int64
Time_Spent_Modules,float64
Participation_Forums,int64
Quiz_Performance_Average,float64
Assignment_Submissions,int64
Resource_Access_Frequency,int64
Session_Duration_Average,float64
Engagement_Level,int64
Hours_Studied,int64
Attendance,int64


In [ ]:
# we re-split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

Model Accuracy: 1.0
              precision    recall  f1-score   support

    Moderate       1.00      1.00      1.00        40

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


array([[40]])

In [ ]:
df["Predicted_Risk"] = model.predict(X)

risk_levels = ["Low", "Moderate", "High"]

dashboard_summary = (
    df["Predicted_Risk"]
    .value_counts()
    .reindex(risk_levels, fill_value=0)
    .reset_index()
)

dashboard_summary.columns = ["Risk_Level", "Count"]

dashboard_summary

df["Predicted_Risk"].value_counts()

,count
Predicted_Risk,
Moderate,198


In [ ]:
def apply_krr(row):
    if row["Attendance"] < 60 and row["Engagement_Level"] <= 1:
        return "High"
    elif row["Attendance"] > 90 and row["Engagement_Level"] >= 3:
        return "Low"
    else:
        return row["Predicted_Risk"]

df["Final_Risk"] = df.apply(apply_krr, axis=1)

df["Final_Risk"].value_counts()

,count
Final_Risk,
Moderate,198


In [ ]:
import joblib

joblib.dump(model, "student_risk_model.pkl")

['student_risk_model.pkl']

In [ ]:
from google.colab import files
files.download("student_risk_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
risk_levels = ["Low", "Moderate", "High"]

dashboard_summary = (
    df["Final_Risk"]
    .value_counts()
    .reindex(risk_levels, fill_value=0)
    .reset_index()
)

dashboard_summary.columns = ["Risk_Level", "Count"]
dashboard_summary

,Risk_Level,Count
0,Low,0
1,Moderate,198
2,High,0


In [ ]:
dashboard_summary.to_csv("dashboard_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("dashboard_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# this cell is used for checking the rows used in our dataset
print("Total rows in df:", df.shape[0])
print(df["Predicted_Risk"].value_counts())
print("Sum:", df["Predicted_Risk"].value_counts().sum())

Total rows in df: 198
Predicted_Risk
Moderate    198
Name: count, dtype: int64
Sum: 198
